# Results for model: meta/llama-4-maverick-17b-128e-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split

def load_data(path):
    return pl.read_parquet(path)

def feature_engineering(df):
    df = df.with_columns([
        (pl.col('feature_00').over(pl.col('date_id')).quantile(0.85)).alias('TOP_QUANTILE')
    ])
    df = df.with_columns([
        (pl.col('feature_00') > pl.col('TOP_QUANTILE')).alias('is_top_quantile')
    ])
    return df

def prepare_data(df):
    df = df.drop_nulls()
    train_df, val_df = df.partition_by(pl.col('date_id') < df['date_id'].median(), as_list=True)
    X_train = train_df.drop(['responder_6', 'date_id'])
    y_train = train_df['responder_6']
    X_val = val_df.drop(['responder_6', 'date_id'])
    y_val = val_df['responder_6']
    return X_train.to_pandas(), y_train.to_pandas(), X_val.to_pandas(), y_val.to_pandas()

def train_model(X_train, y_train, X_val, y_val):
    model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=5, verbose=False)
    return model

def main():
    data_path = './jane_street_data/train.parquet'
    df = load_data(data_path)
    df = feature_engineering(df)
    X_train, y_train, X_val, y_val = prepare_data(df)
    model = train_model(X_train, y_train, X_val, y_val)

if __name__ == "__main__":
    main()